# Investigate related waveform intervals

**Import → select → search → inspect.** This notebook uses MorletDB's PostgreSQL
recording APIs through the existing thin Python client. NumPy/FFT below are for
visual inspection only; PostgreSQL performs extraction and ranking.

The default archive is **synthetic**, with an exact repeated transient, a different
background recording, and a deliberately misleading sign-flipped transient.
This is a reproducible API demonstration, not a relevance benchmark or fault diagnosis.

Before running: follow [setup](README.md), use an isolated demo database, set
`DATABASE_URL`, and select **Run All**. Setup creates `morlet_notebook.signals`
and persists demo recordings; rerunning verifies their committed contents.


In [ ]:
import html
import json
import os
from pathlib import Path
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import psycopg
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import HTML, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/client/morlet_client.py').is_file())
sys.path[:0] = [str(ROOT / 'tools/client'), str(ROOT / 'tools/ingest'),
                str(ROOT / 'examples')]
from morlet_client import RecordingClient
from parquet_import import Config, import_files
from inspect_recordings import archive

# Standard PG* variables also work. Never print a connection string.
DSN = os.environ.get('DATABASE_URL', '')
COLLECTION = 'morlet_notebook.signals'
QUERY_ID, START, END = 'query', 768, 1280
COHORT = {'dataset': 'visual-notebook-v1'}
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})


## 1. Import the demo archive

This cell writes only the isolated demo collection above. It requires an installed
`pg_morlet` extension and permission to create a schema/collection. Install the
extension separately using the setup commands; this notebook does not elevate roles.

The example reuses the existing archive generator and Parquet importer, with
300-sample chunks so the selected 512-sample intervals cross chunk boundaries.
Every recording retains explicit rate, channel, units and acquisition metadata.


In [ ]:
with psycopg.connect(DSN) as conn:
    conn.execute('CREATE SCHEMA IF NOT EXISTS morlet_notebook')
    if conn.execute("SELECT to_regclass('morlet_notebook.signals')").fetchone()[0] is None:
        conn.execute("SELECT morlet_create_stream_collection('morlet_notebook','signals')")

with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / 'signals.parquet'
    archive(path)
    rows = pq.read_table(path).to_pylist()
    # Copy the related recording, reversing only the transient's polarity.
    flipped = dict(next(row for row in rows if row['recording_id'] == 'related'))
    flipped['recording_id'] = 'sign_flipped'
    flipped['samples'] = list(flipped['samples'])
    flipped['samples'][1536:2048] = [-x for x in flipped['samples'][1536:2048]]
    rows.append(flipped)
    rows.sort(key=lambda row: row['recording_id'])
    for row in rows:
        row['metadata'] = json.dumps({**COHORT, 'synthetic': True,
                                      'asset': 'demo-sensor',
                                      'scenario': row['recording_id']})
    pq.write_table(pa.Table.from_pylist(rows), path, compression='zstd')
    config = Config(chunk_size=300, rpc_chunks=4, commit_chunks=8)
    report = import_files([path], COLLECTION, config, dsn=DSN)
print('Archive import:', report['status'], '| synthetic recordings:', len(rows))


## 2. Select an interval and search

Coordinates are zero-based, half-open sample intervals. Acquisition compatibility
and metadata filtering happen in PostgreSQL. Source exclusion and greedy overlap
suppression run before top-k; distances are **not confidence scores**.

The query contains a transient at `[768,1280)`. The related recording contains the
same samples at `[1536,2048)`. A second recording contains their negative.


In [ ]:
with psycopg.connect(DSN) as conn:
    # Bound execution time without changing connection credentials or roles.
    conn.execute("SET LOCAL statement_timeout = '30s'")
    client = RecordingClient(conn, COLLECTION)
    query = client.read_interval(QUERY_ID, START, END)
    matches = client.search_interval(QUERY_ID, START, END, metadata=COHORT, top_k=5)
    inspected = [client.read_interval(m['recording_id'], m['start_sample'], m['end_sample'])
                 for m in matches]
assert matches, 'No compatible candidates; check collection, cohort and window settings.'
print(f"Query: {QUERY_ID} [{START},{END}) | {query['sample_rate_hz']} Hz | "
      f"{query['channel']} | {query['unit']}")


## 3. Inspect rank, provenance and aligned correlation

Correlation is calculated here only as an explanatory, aligned time-domain
diagnostic. It is not used to rank results. Undefined correlation for a constant
interval is shown as `n/a`. Metadata travels with each retrieved raw interval.


In [ ]:
def correlation(left, right):
    left, right = np.asarray(left, dtype=float), np.asarray(right, dtype=float)
    if np.std(left) == 0 or np.std(right) == 0:
        return None
    return float(np.corrcoef(left, right)[0, 1])

headers = ['Rank', 'Recording', 'Samples [start,end)', 'Distance ↓',
           'Aligned correlation', 'Rate / channel / unit', 'Metadata']
table = ['<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr>']
for rank, (match, row) in enumerate(zip(matches, inspected), 1):
    corr = correlation(query['samples'], row['samples'])
    values = [rank, match['recording_id'], f"[{match['start_sample']},{match['end_sample']})",
              f"{match['distance']:.6g}", 'n/a' if corr is None else f'{corr:+.3f}',
              f"{row['sample_rate_hz']} / {row['channel']} / {row['unit']}",
              json.dumps(row['metadata'], sort_keys=True)]
    table.append('<tr>' + ''.join(f'<td>{html.escape(str(v))}</td>' for v in values) + '</tr>')
display(HTML(''.join(table) + '</table>'))


## 4. Compare time and frequency

Time axes are relative to each interval's start, so events are visually aligned.
Original recording coordinates remain in the titles/table. Frequency plots show
the magnitude of the demeaned, Hann-windowed FFT; they are inspection plots,
not the Morlet representation used by the database. All time panels share a
vertical range so amplitude and polarity remain visible.


In [ ]:
panels = [('QUERY', query, None)] + [
    (f"MATCH {i}", row, match['distance'])
    for i, (match, row) in enumerate(zip(matches, inspected), 1)]
fig, axes = plt.subplots(len(panels), 2, figsize=(12, 2.5 * len(panels)), squeeze=False)
limit = max(np.max(np.abs(row['samples'])) for _, row, _ in panels) * 1.12 or 1.0
for (label, row, distance), (time_ax, freq_ax) in zip(panels, axes):
    samples = np.asarray(row['samples'], dtype=float)
    rate = float(row['sample_rate_hz'])
    time_ax.plot(np.arange(len(samples)) / rate * 1000, samples, color='#087f8c', lw=1.1)
    time_ax.set_ylim(-limit, limit)
    time_ax.set(xlabel='Time from interval start (ms)', ylabel=f"Amplitude ({row['unit']})")
    suffix = '' if distance is None else f' · distance {distance:.4g}'
    time_ax.set_title(f"{label} · {row['recording_id']} [{row['start_sample']},{row['end_sample']}){suffix}", loc='left', fontsize=9)
    spectrum = np.abs(np.fft.rfft((samples - samples.mean()) * np.hanning(len(samples))))
    freq_ax.semilogy(np.fft.rfftfreq(len(samples), 1 / rate), np.maximum(spectrum, 1e-12),
                     color='#555c75', lw=1.1)
    freq_ax.set(xlabel='Frequency (Hz)', ylabel='FFT magnitude (arbitrary scale)',
                title=f"{row['sample_rate_hz']} Hz · {row['channel']} · {row['unit']}")
fig.tight_layout()
display(fig)
plt.close(fig)


## 5. Check the misleading match

Negating every sample preserves pooled Morlet energies and the stored statistical
features. The sign-flipped transient therefore has essentially zero feature
distance despite opposite polarity. Whether polarity matters depends on the
investigation. Similar energy is not proof of the same physical event.

Assertions below are deterministic demo contracts, not measured search accuracy.
They also verify raw reconstruction across acquisition chunks.


In [ ]:
hits = {(m['recording_id'], m['start_sample']): (m, row)
        for m, row in zip(matches, inspected)}
exact, exact_raw = hits[('related', 1536)]
flipped, flipped_raw = hits[('sign_flipped', 1536)]
assert exact['distance'] < 1e-10 and flipped['distance'] < 1e-10
np.testing.assert_array_equal(exact_raw['samples'], query['samples'])
np.testing.assert_array_equal(flipped_raw['samples'], -np.asarray(query['samples']))
assert correlation(query['samples'], flipped_raw['samples']) < -0.999999
print('Verified: exact repeat, polarity distractor, provenance and cross-chunk reads.')


## Investigate your own recordings

Import approved, local immutable Parquet with the [existing importer](../../tools/ingest/README.md).
Then skip the demo import cell and set `COLLECTION`, `QUERY_ID`, `START`, `END`
and `COHORT` in the configuration cell. Run sections 2–4; skip the synthetic-only
assertions in section 5. The interval length must match your collection's extraction
window. Rate/channel/unit/window/normalization/extractor compatibility is required.

The default exact-search budget is 4096 eligible segments; a broad cohort can fail
that guardrail. There is no resampling or speed invariance. Do not interpret top-k
as an acceptance threshold. See [real-data evaluation](../../evaluation/README.md)
for the current CWRU baseline and its limitations.

Connections close at each cell boundary; demo records remain for replay. To remove
only this demo, deliberately run `DROP SCHEMA morlet_notebook CASCADE` in your
isolated database. Do not run that against a schema containing valuable data.
Notebook outputs may contain private recordings/metadata: clear them before sharing.
The committed notebook intentionally contains no outputs or credentials.
